[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/polars-certified/notebooks/day-03-lazy-evaluation-and-query-optimization.ipynb#scrollTo=1a2b3c4d)

---
# Day 3 · Lazy Evaluation and Query Optimization
**certified-journeys / polars-certified** &nbsp;|&nbsp; Performance

> **Goal for today:** Understand how LazyFrame defers execution, how Polars optimises query plans, and when to use lazy vs eager.

---
## Eager vs Lazy — the mental model

| | Eager (`DataFrame`) | Lazy (`LazyFrame`) |
|---|---|---|
| Execution | Immediately on each call | Deferred until `.collect()` |
| Optimization | None — runs as written | Full query plan optimization |
| Memory | Materializes every step | Reads only what's needed |
| Entry points | `pl.DataFrame()`, `pl.read_csv()` | `pl.LazyFrame()`, `pl.scan_csv()` |
| Best for | Exploration, small data | Production pipelines, large data |

When you use a `LazyFrame`, Polars builds a **logical query plan** — a tree of operations. Before executing, the optimizer rewrites the plan to:
1. **Predicate pushdown** — move filters as early as possible (read less data)
2. **Projection pushdown** — read only the columns you actually use
3. **Common subexpression elimination** — avoid computing the same thing twice

> **Tip:** Always prefer `LazyFrame` for production pipelines. Polars will optimize the query plan before executing.

In [ ]:
%pip install -q polars numpy

---
## Step 1 · Create a LazyFrame

A `LazyFrame` is created from a `DataFrame` with `.lazy()`, or directly from files with `scan_csv()` / `scan_parquet()`.
No data is read or computed until `.collect()` is called.

In [ ]:
import polars as pl
import time
import random
import csv

# --- Generate and write a CSV for scan_csv() demos ---
random.seed(0)
categories = ["A", "B", "C", "D"]
regions    = ["north", "south", "east", "west"]
rows = [
    {
        "id":       i,
        "category": random.choice(categories),
        "region":   random.choice(regions),
        "value":    round(random.uniform(1.0, 1000.0), 2),
        "quantity": random.randint(1, 50),
        "flag":     random.choice([True, False]),
    }
    for i in range(1, 500_001)    # 500 000 rows
]
CSV_PATH = "/tmp/large_data.csv"
with open(CSV_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)
print(f"Written {len(rows):,} rows to {CSV_PATH}")

In [ ]:
# --- Three ways to get a LazyFrame ---

# 1. From an eager DataFrame
df_eager = pl.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
lf1 = df_eager.lazy()              # convert eager → lazy
print("Type of lf1:", type(lf1))  # polars.LazyFrame

# 2. From a constructor
lf2 = pl.LazyFrame({"a": [10, 20], "b": ["x", "y"]})

# 3. From a file — scan_csv does NOT read the file yet
lf3 = pl.scan_csv(CSV_PATH)        # no I/O happens here
print("LazyFrame from scan_csv — no data loaded yet")
print("Schema known without reading:", lf3.schema)  # schema inferred from header only

# collect() triggers execution and returns a DataFrame
result = lf3.filter(pl.col("category") == "A").limit(5).collect()
print("\nFirst 5 rows of category A:")
print(result)

**What just happened?**
- `scan_csv()` returned a `LazyFrame` **instantly** — it read only the header row to infer schema
- The filter and limit were **not executed** until `.collect()` was called
- `.schema` is available on a `LazyFrame` without loading data — Polars reads just the header
- **`lazy()` on an existing DataFrame** is useful to opt into query optimization for a chain of operations
- `limit(5)` in lazy mode tells the optimizer to stop reading once 5 rows are collected — very efficient

---
## Step 2 · Read the query plan with `.explain()`

`.explain()` prints the **optimized query plan** — the tree of operations Polars will actually execute.
Reading query plans helps you understand what the optimizer did and spot inefficiencies.

In [ ]:
lf = pl.scan_csv(CSV_PATH)

# Build a query: filter, select two columns, group by
query = (
    lf
    .filter(pl.col("category") == "A")             # predicate
    .select(pl.col("category", "region", "value"))  # projection
    .group_by("region")
    .agg(pl.col("value").mean().alias("avg_value"))
)

# explain() — see the optimized plan (no data loaded yet)
print("=== Optimized query plan ===")
print(query.explain())                 # default: show optimized plan

print("\n=== Unoptimized query plan (for comparison) ===")
print(query.explain(optimized=False))  # what you wrote, before optimization

**What just happened?**
- The **optimized plan** shows operations re-ordered for efficiency — filter and projection happen at the scan level
- The **unoptimized plan** reflects the order you wrote operations — it's what a naive executor would do
- Look for `SELECTION` near the bottom of the plan — that means **predicate pushdown** is active
- Look for column lists in the CSV scan node — that means **projection pushdown** is active (only listed columns are read)
- **Reading query plans is a key debugging skill** — when a query is slow, inspect `.explain()` first

---
## Step 3 · Predicate pushdown

**Predicate pushdown** moves filter conditions to the data source — Polars reads only the rows
that satisfy the filter, instead of reading everything and then filtering.

Compare the query plans when you filter early vs late:

In [ ]:
lf = pl.scan_csv(CSV_PATH)

# --- Query A: filter FIRST, then group_by ---
query_a = (
    lf
    .filter(pl.col("value") > 500)    # filter early in the chain
    .group_by("category")
    .agg(pl.col("value").sum())
)

# --- Query B: group_by FIRST, then filter (logically different, but shows plan difference) ---
# Note: we filter on a pre-aggregate column to make the example comparable
query_b = (
    lf
    .group_by("category")
    .agg(pl.col("value").sum())
    .filter(pl.col("value") > 500_000) # filter AFTER aggregation (different semantic)
)

print("=== Query A plan (filter before group_by — pushdown candidate) ===")
print(query_a.explain())

print("\n=== Query B plan (filter after aggregation — cannot push down) ===")
print(query_b.explain())

# In Query A, notice the filter appears in the CSV SCAN node — pushed all the way down
# In Query B, the filter stays after GROUP_BY — it cannot be pushed past an aggregation

**What just happened?**
- In Query A, the optimizer pushed the `value > 500` filter **into the CSV scan** — Polars skips rows while reading
- In Query B, the filter is on the aggregated `value` — it **cannot be pushed** past the `group_by`, so it stays after
- **The practical implication:** write your filters early in the chain even in lazy mode — the optimizer will push them down, and if it can't, your intention is still clear
- Predicate pushdown is especially powerful with Parquet files, which store row-group statistics that Polars uses to skip entire chunks
- **Column names in `.explain()` output** show you exactly which columns are being read at the source level

---
## Step 4 · Projection pushdown

**Projection pushdown** ensures that only the columns you actually use are read from the source.
For wide CSVs and Parquet files this can cut I/O by 80%+.

In [ ]:
lf = pl.scan_csv(CSV_PATH)

# Query that only uses 2 of the 6 columns
query = (
    lf
    .select(pl.col("category", "value"))   # only two columns selected
    .group_by("category")
    .agg(pl.col("value").mean().alias("avg"))
)

print("=== Plan with projection pushdown ===")
print(query.explain())
# Notice: the CSV SCAN node lists only [category, value] — the other 4 columns are never read

# Without any projection — all columns read
query_all = lf.group_by("category").agg(pl.col("value").mean())
print("\n=== Plan without explicit projection ===")
print(query_all.explain())
# Polars still pushes down the projection for used columns — only category + value are read

**What just happened?**
- Even without an explicit `select()`, Polars tracked which columns are needed and pushed projection down
- The CSV scan node shows only the columns that are actually used downstream — `id`, `region`, `quantity`, `flag` are never read
- **For Parquet files this is even more powerful** — Parquet stores columns separately, so unneeded columns are skipped at OS I/O level
- Projection pushdown is automatic — you get it for free with any `LazyFrame` / `scan_*` entry point
- **Combine with predicate pushdown** for maximum efficiency: filter early, select only needed columns

---
## Step 5 · `.collect()` vs `.collect(streaming=True)` and timing

Polars has two execution modes for `collect()`:
- Default: loads the entire result into RAM
- `streaming=True`: processes data in chunks — use for datasets larger than RAM

In [ ]:
lf = pl.scan_csv(CSV_PATH)

query = (
    lf
    .filter(pl.col("value") > 200)
    .group_by("category", "region")
    .agg(
        pl.col("value").sum().alias("total_value"),
        pl.col("quantity").mean().alias("avg_qty"),
    )
)

# --- Default collect: materializes all results in RAM ---
t0 = time.perf_counter()
result_default = query.collect()
t1 = time.perf_counter()
print(f"Default collect:    {t1-t0:.3f}s — {len(result_default)} rows")

# --- Streaming collect: processes in chunks (lower peak memory) ---
# streaming=True is useful when the intermediate data is larger than RAM
# Not all operations support streaming — Polars falls back to default if needed
t0 = time.perf_counter()
result_streaming = query.collect(engine="streaming")  # Polars ≥ 0.19
t1 = time.perf_counter()
print(f"Streaming collect:  {t1-t0:.3f}s — {len(result_streaming)} rows")

# Results are identical — streaming is a memory strategy, not a logic change
print("\nResults match:", result_default.sort(["category","region"]).frame_equal(
    result_streaming.sort(["category","region"])
))

**What just happened?**
- Both `collect()` and `collect(engine="streaming")` produce the same result — streaming is a **memory strategy**
- **Streaming** processes the file in batches, keeping only the current batch in RAM — essential for out-of-core data
- Not all operations support streaming (some aggregations require full data) — Polars warns you if it falls back
- For this 500k-row example, default collect is often faster (no batching overhead) — streaming shines at GB/TB scale
- **Rule of thumb:** use streaming when your data is larger than ~50% of available RAM

---
## Challenge — do this yourself

Below is an eager pipeline with 3 operations. Your tasks:
1. Convert it to a lazy pipeline using `scan_csv()` and `.collect()`
2. Print the `.explain()` output and identify where predicate pushdown and projection pushdown appear
3. Confirm the results are identical

In [ ]:
# --- Eager version (given) ---
eager_result = (
    pl.read_csv(CSV_PATH)                     # reads entire file into RAM
    .filter(pl.col("region") == "north")       # filter after full load
    .select(pl.col("category", "region", "value", "quantity"))  # select cols
    .group_by("category")
    .agg(
        pl.col("value").sum().alias("total"),
        pl.col("quantity").sum().alias("units"),
    )
    .sort("category")
)
print("Eager result:")
print(eager_result)

# --- TODO: rewrite as lazy pipeline ---
lazy_result = (
    # pl.scan_csv(CSV_PATH)
    # ...
    # .collect()
    None  # replace this
)

# TODO: print .explain() on your lazy query (before .collect())

# TODO: verify results match
# print(eager_result.frame_equal(lazy_result.sort("category")))

---
## Day 3 key concepts recap

| Concept | What to remember |
|---|---|
| `LazyFrame` | Defers execution — Polars builds a query plan instead of running immediately |
| `scan_csv()` / `scan_parquet()` | File-scanning entry points — no I/O until `.collect()` |
| `.explain()` | Prints the optimized (or raw) query plan — essential debugging tool |
| Predicate pushdown | Filters pushed to the source — reads fewer rows |
| Projection pushdown | Only needed columns read from source — reduces I/O |
| `.collect()` | Triggers execution, returns an eager `DataFrame` |
| `streaming=True` | Processes data in chunks — use when data exceeds available RAM |

> **Tip:** Always prefer `LazyFrame` for production pipelines. Polars will optimize the query plan before executing — you get predicate and projection pushdown for free.

---
## What's next

**Day 4** → GroupBy, aggregations and window functions: `group_by().agg()`, `over()`, `rolling()`, `ewm_mean()`, and `partition_by()`.

Mark Day 3 complete in your [tracker](../index.html).